In [10]:
# Import libraries and packages
import pandas as pd
import numpy as np

import json
import torch
import pickle
import joblib

import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             f1_score, confusion_matrix, roc_curve,
                             matthews_corrcoef)

In [7]:
# Load saved artifacts
with open('saved/col_lists.json', 'r') as f:
    cols = json.load(f)

demo_cols    = cols['demo_cols']
disease_cols = cols['disease_cols']
demo_meta    = cols['demo_meta']
disease_meta = cols['disease_meta']

# Load MTL test predictions
test_preds   = np.load('saved/mtl_test_preds.npy')
test_targets = np.load('saved/mtl_test_targets.npy')

# Quick shape check
print(f"Test preds shape:   {test_preds.shape}") # expect (1091, 15)
print(f"Test targets shape: {test_targets.shape}") # expect (1091, 15)


Test preds shape:   (1091, 15)
Test targets shape: (1091, 15)


In [ ]:
# Metric functions
def optimal_threshold(t, p):
    fpr, tpr, thresholds = roc_curve(t, p)
    youden_j = tpr - fpr
    return thresholds[np.argmax(youden_j)]

def print_metrics(all_targets, all_preds, disease_cols, disease_meta, threshold_method='youden'):
    if threshold_method == 'youden':
        print("Threshold: Youden's J (maximises sensitivity + specificity)")
    else:
        print("Threshold: 0.5 (standard literature default)")

    print(f"\n{'Disease':30s}  {'PRAUC':>6}  {'AUROC':>6}  {'Sens':>6}  {'Spec':>6}  {'F1':>6}  {'MCC':>6}  {'Thresh':>6}  {'Prev':>6}")
    print("-" * 110)

    for k, col in enumerate(disease_cols):
        valid = ~np.isnan(all_targets[:, k])
        if valid.sum() == 0:
            continue

        t = all_targets[valid, k]
        p = all_preds[valid, k]
        prev = t.mean()

        thresh = optimal_threshold(t, p) if threshold_method == 'youden' else 0.5
        pred_binary = (p >= thresh).astype(int)

        prauc = average_precision_score(t, p)
        auroc = roc_auc_score(t, p)
        f1    = f1_score(t, pred_binary, zero_division=0)
        mcc   = matthews_corrcoef(t, pred_binary)

        tn, fp, fn, tp = confusion_matrix(t, pred_binary, labels=[0,1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        print(f"  {disease_meta[col]:30s}  {prauc:6.3f}  {auroc:6.3f}  {sensitivity:6.3f}  {specificity:6.3f}  {f1:6.3f}  {mcc:6.3f}  {thresh:6.3f}  {prev:6.3f}")

    print("-" * 110)

# MTL Model Results
print("=" * 110)
print("MTL MODEL — TEST SET RESULTS")
print("=" * 110)
print_metrics(test_targets, test_preds, disease_cols, disease_meta, threshold_method='youden')
print("\n")
print_metrics(test_targets, test_preds, disease_cols, disease_meta, threshold_method='fixed')

MTL MODEL — TEST SET RESULTS
Threshold: Youden's J (maximises sensitivity + specificity)

Disease                          PRAUC   AUROC    Sens    Spec      F1     MCC  Thresh    Prev
--------------------------------------------------------------------------------------------------------------
  Diabetes                         0.454   0.876   0.902   0.688   0.390   0.371   0.087   0.103
  Pre-diabetes                     0.350   0.785   0.663   0.768   0.387   0.307   0.183   0.116
  Hypertension                     0.747   0.860   0.829   0.763   0.728   0.566   0.394   0.346
  High cholesterol                 0.703   0.818   0.731   0.772   0.692   0.494   0.501   0.374
  Arthritis                        0.642   0.814   0.851   0.638   0.645   0.454   0.329   0.315
  Congestive heart failure         0.385   0.927   0.880   0.821   0.226   0.296   0.019   0.029
  Coronary heart disease           0.331   0.869   0.878   0.699   0.225   0.263   0.017   0.048
  Heart attack           

In [12]:
# ── Load data ─────────────────────────────────────────────────────────────────
data = pd.read_csv('saved/nhanes_clean.csv')
scaler = joblib.load('saved/demo_scaler.pkl')

continuous_cols = [
    'RIDAGEYR', 'BMXBMI', 'BMXWAIST',
    'ALQ121', 'PAQ706', 'OCQ180', 'HUQ010',
    'SBP', 'DBP'
]
data[continuous_cols] = scaler.transform(data[continuous_cols])
data[demo_cols]    = data[demo_cols].astype(float)
data[disease_cols] = data[disease_cols].astype(float)

train_df, temp_df = train_test_split(data, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

class NHANESDataset(Dataset):
    def __init__(self, df, demo_cols, disease_cols):
        self.X = torch.tensor(df[demo_cols].astype(float).values, dtype=torch.float32)
        self.D = torch.tensor(df[disease_cols].astype(float).values, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.D[idx]

test_loader = DataLoader(NHANESDataset(test_df, demo_cols, disease_cols), 
                         batch_size=64, shuffle=False)

# ── Load model ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('saved/mtl_model.pt', map_location=device)
model = MaskedMTLNet(
    d_demo=checkpoint['d_demo'], K=checkpoint['K'],
    H=checkpoint['H'], dropout=checkpoint['dropout'],
    circular_pairs=checkpoint['circular_pairs']
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# ── Compute importance ────────────────────────────────────────────────────────
all_features = demo_cols + disease_cols

all_X, all_D = [], []
with torch.no_grad():
    for X_batch, D_batch in test_loader:
        all_X.append(X_batch)
        all_D.append(D_batch)
all_X = torch.cat(all_X).to(device)
all_D = torch.cat(all_D).to(device)

with torch.no_grad():
    base_preds = model(all_X, all_D).cpu().numpy()
targets = all_D.cpu().numpy()

importance = {}
for k, col in enumerate(disease_cols):
    valid = ~np.isnan(targets[:, k])
    t = targets[valid, k]
    base_auroc = roc_auc_score(t, base_preds[valid, k])
    imps = []
    for feat in all_features:
        drops = []
        for _ in range(5):
            X_perm = all_X.clone()
            D_perm = all_D.clone()
            idx = torch.randperm(all_X.shape[0])
            if feat in demo_cols:
                j = demo_cols.index(feat)
                X_perm[:, j] = X_perm[idx, j]
            else:
                j = disease_cols.index(feat)
                if j != k:
                    D_perm[:, j] = D_perm[idx, j]
            with torch.no_grad():
                perm_preds = model(X_perm, D_perm).cpu().numpy()
            drops.append(base_auroc - roc_auc_score(t, perm_preds[valid, k]))
        imps.append(np.mean(drops))
    importance[col] = dict(zip(all_features, imps))

# ── Plot heatmap ──────────────────────────────────────────────────────────────
input_labels  = [demo_meta.get(c, c) for c in demo_cols] + \
                [disease_meta.get(c, c) for c in disease_cols]
target_labels = [disease_meta[c] for c in disease_cols]
n_demo = len(demo_cols)

full_matrix = np.zeros((len(disease_cols), len(all_features)))
for k, tc in enumerate(disease_cols):
    for j, ic in enumerate(all_features):
        full_matrix[k, j] = importance[tc].get(ic, 0)
        if ic == tc:
            full_matrix[k, j] = np.nan

fig, ax = plt.subplots(figsize=(22, 11))
im = ax.imshow(full_matrix, cmap='YlOrRd', aspect='auto',
               vmin=0, vmax=np.nanmax(full_matrix))
ax.set_xticks(range(len(all_features)))
ax.set_yticks(range(len(disease_cols)))
ax.set_xticklabels(input_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(target_labels, fontsize=8)
ax.axvline(x=n_demo - 0.5, color='#1a1a2e', linewidth=2.5, linestyle='--')
ax.text(n_demo / 2 - 0.5, -1.8, 'Demographics', ha='center',
        fontsize=9, fontweight='bold', color='#2c7a4b')
ax.text(n_demo + len(disease_cols) / 2 - 0.5, -1.8, 'Disease flags', ha='center',
        fontsize=9, fontweight='bold', color='#c0392b')
ax.set_xlabel('Input feature (driver)', fontsize=11, labelpad=30)
ax.set_ylabel('Target disease (being predicted)', fontsize=11, labelpad=10)
ax.set_title('Permutation Importance — Demographics and Disease Flags → Disease Targets\n'
             '(AUROC drop when feature is permuted)', fontsize=12, fontweight='bold', pad=15)
for i in range(len(disease_cols)):
    for j in range(len(all_features)):
        val = full_matrix[i, j]
        if not np.isnan(val) and val > 0.005:
            text_color = 'white' if val > np.nanmax(full_matrix) * 0.6 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=6, color=text_color)
plt.colorbar(im, ax=ax, label='Importance (AUROC drop when permuted)', shrink=0.7)
plt.tight_layout()
plt.show()

ValueError: could not convert string to float: "b'21:30'"

In [ ]:
# ── Full Importance Heatmap: Demographics + Diseases → Disease Targets ────────
import matplotlib.pyplot as plt
import numpy as np

all_input_features = demo_cols + disease_cols
input_labels = [demo_meta.get(col, col) for col in demo_cols] + \
               [disease_meta.get(col, col) for col in disease_cols]
target_labels = [disease_meta[col] for col in disease_cols]

# Build full matrix: rows = targets, cols = all inputs
full_matrix = np.zeros((len(disease_cols), len(all_input_features)))

for k, target_col in enumerate(disease_cols):
    for j, input_col in enumerate(all_input_features):
        if input_col in importance[target_col]:
            full_matrix[k, j] = importance[target_col][input_col]

# Mask diagonal where input == target
for k, target_col in enumerate(disease_cols):
    for j, input_col in enumerate(all_input_features):
        if input_col == target_col:
            full_matrix[k, j] = np.nan

# ── Plot ──────────────────────────────────────────────────────────────────────
n_inputs = len(all_input_features)
n_targets = len(disease_cols)
n_demo = len(demo_cols)

fig, ax = plt.subplots(figsize=(22, 11))

im = ax.imshow(full_matrix, cmap='YlOrRd', aspect='auto',
               vmin=0, vmax=np.nanmax(full_matrix))

# Axis ticks
ax.set_xticks(range(n_inputs))
ax.set_yticks(range(n_targets))
ax.set_xticklabels(input_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(target_labels, fontsize=8)

# Vertical divider between demographics and diseases
ax.axvline(x=n_demo - 0.5, color='#1a1a2e', linewidth=2.5, linestyle='--')
ax.text(n_demo / 2 - 0.5, -1.8, 'Demographics', ha='center', fontsize=9,
        fontweight='bold', color='#2c7a4b')
ax.text(n_demo + len(disease_cols) / 2 - 0.5, -1.8, 'Disease flags', ha='center',
        fontsize=9, fontweight='bold', color='#c0392b')

ax.set_xlabel('Input feature (driver)', fontsize=11, labelpad=30)
ax.set_ylabel('Target disease (being predicted)', fontsize=11, labelpad=10)
ax.set_title('Permutation Importance — Demographics and Disease Flags → Disease Targets\n'
             '(AUROC drop when feature is permuted)',
             fontsize=12, fontweight='bold', pad=15)

# Value annotations — only show values > 0.005 to avoid clutter
for i in range(n_targets):
    for j in range(n_inputs):
        val = full_matrix[i, j]
        if not np.isnan(val) and val > 0.005:
            text_color = 'white' if val > np.nanmax(full_matrix) * 0.6 else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=6, color=text_color)

plt.colorbar(im, ax=ax, label='Importance (AUROC drop when permuted)', shrink=0.7)
plt.tight_layout()
plt.show()